In [ ]:
# SEN12MS-CR-TS Cloud Removal: EDA, Baseline, Multi-Temporal U-Net
# Author: Muhammad Abdul Rafi
# Dataset target: asiaWest_n from SEN12MS-CR-TS

# SEN12MS-CR-TS Cloud Removal

Notebook ini disusun ulang untuk `asiaWest_n`.

Alur utama:
1. Validasi struktur dataset multi-temporal.
2. Preprocessing Sentinel-2 + Sentinel-1.
3. Pseudo cloud mask dari S2.
4. Target awal: clear composite berbobot, bukan temporal median polos.
5. Baseline: cloudy t0, best-time, dan weighted composite.
6. Model: multi-temporal ResUNet ringan.

Catatan penting:
- Dataset ini tidak sama seperti RICE yang punya label clear eksplisit.
- Jadi tahap awal memakai pseudo-target dari beberapa tanggal.
- Metrik utama bukan accuracy, tapi MAE pada area awan (`cloud_mae`).

In [ ]:
import os
import re
import math
import random
import zipfile
import json
from datetime import datetime
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.ndimage import gaussian_filter
except Exception:
    gaussian_filter = None

try:
    import tifffile
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile"])
    import tifffile

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 1. Konfigurasi

In [ ]:
@dataclass
class CFG:
    DATASET_NAME: str = "asiaWest_n"
    LOCAL_ROOT: Path = Path(r"C:/Users/Vulpolish/Downloads/Tugas Akhir/asiaWest_n")
    KAGGLE_INPUT: Path = Path("/kaggle/input")
    WORK_DATA: Path = Path("/kaggle/working/data")
    OUT_DIR: Path = Path("/kaggle/working/sen12ms_outputs") if Path("/kaggle/working").exists() else Path("sen12ms_outputs")
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 2
    EPOCHS: int = 50
    LR: float = 1e-4
    TRAIN_RATIO: float = 0.80
    VAL_RATIO: float = 0.10
    RUN_TRAINING: bool = True
    USE_S1: bool = True
    BASE_CHANNELS: int = 32
    CLOUD_THRESHOLD: float = 0.22
    MASK_LOW: float = 0.08
    MASK_HIGH: float = 0.20
    TARGET_SHARPNESS: float = 10.0
    MAX_TRAIN_SAMPLES: int = 0  # full dataset
    HEAVY_CLOUD_OVERSAMPLE: bool = True
    HEAVY_CLOUD_THRESHOLD: float = 0.30
    HEAVY_PIXEL_THRESHOLD: float = 0.66
    HEAVY_LOSS_BOOST: float = 2.5
    HEAVY_BINARY_BOOST: float = 2.0

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.WORK_DATA.mkdir(parents=True, exist_ok=True)
CFG

## 2. Cari Dataset dan Unzip Jika Perlu

In [ ]:
def unzip_if_needed(input_root: Path, work_root: Path, dataset_name: str = "asiaWest_n") -> None:
    if not input_root.exists():
        return
    if (work_root / dataset_name).exists():
        return
    zips = sorted(input_root.rglob(f"{dataset_name}.zip")) + sorted(input_root.rglob("*.zip"))
    if not zips:
        return
    zip_path = zips[0]
    print("Unzipping:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(work_root)


def find_dataset_root() -> Path:
    candidates = []
    if CFG.LOCAL_ROOT.exists():
        candidates.append(CFG.LOCAL_ROOT)
    if CFG.KAGGLE_INPUT.exists():
        candidates += list(CFG.KAGGLE_INPUT.rglob(CFG.DATASET_NAME))
    candidates += list(CFG.WORK_DATA.rglob(CFG.DATASET_NAME))
    for c in candidates:
        if c.is_dir() and any(c.glob("ROIs*")):
            return c
    unzip_if_needed(CFG.KAGGLE_INPUT, CFG.WORK_DATA, CFG.DATASET_NAME)
    candidates = list(CFG.WORK_DATA.rglob(CFG.DATASET_NAME))
    for c in candidates:
        if c.is_dir() and any(c.glob("ROIs*")):
            return c
    print("Kaggle input folders:")
    if CFG.KAGGLE_INPUT.exists():
        for p in CFG.KAGGLE_INPUT.glob("*"):
            print(" -", p)
    raise FileNotFoundError("Dataset asiaWest_n belum ditemukan. Upload/attach asiaWest_n.zip atau folder asiaWest_n dulu.")

DATA_ROOT = find_dataset_root()
print("DATA_ROOT:", DATA_ROOT)
print("Top folders:", [p.name for p in DATA_ROOT.iterdir() if p.is_dir()][:10])

## 3. Index SEN12MS-CR-TS

In [ ]:
SEN12_RE = re.compile(
    r"^(?P<sensor>s[12])_(?P<roi_group>ROIs\d+)_(?P<roi_id>\d+)_ImgNo_"
    r"(?P<time_id>\d+)_(?P<date>\d{4}-\d{2}-\d{2})_patch_(?P<patch_id>\d+)\.tif$",
    re.IGNORECASE,
)

@dataclass(frozen=True)
class SEN12Sample:
    roi_group: str
    roi_id: str
    patch_id: int
    s1: Dict[int, Path]
    s2: Dict[int, Path]
    dates: Dict[Tuple[str, int], str]

    @property
    def sample_id(self) -> str:
        return f"{self.roi_group}_{self.roi_id}_patch_{self.patch_id}"


def build_index(root: Path) -> List[SEN12Sample]:
    grouped = {}
    for path in sorted(root.rglob("*.tif")):
        m = SEN12_RE.match(path.name)
        if not m:
            continue
        gd = m.groupdict()
        key = (gd["roi_group"], gd["roi_id"], int(gd["patch_id"]))
        sensor = gd["sensor"].upper()
        time_id = int(gd["time_id"])
        item = grouped.setdefault(key, {"S1": {}, "S2": {}, "dates": {}})
        item[sensor][time_id] = path
        item["dates"][(sensor, time_id)] = gd["date"]

    samples = []
    for (roi_group, roi_id, patch_id), item in sorted(grouped.items()):
        if set(item["S1"]) == {0,1,2,3} and set(item["S2"]) == {0,1,2,3}:
            samples.append(SEN12Sample(roi_group, roi_id, patch_id, dict(sorted(item["S1"].items())), dict(sorted(item["S2"].items())), dict(sorted(item["dates"].items()))))
    return samples

samples = build_index(DATA_ROOT)
print("Complete multi-temporal samples:", len(samples))
print(samples[0])

In [ ]:
rows = []
for s in samples:
    rows.append({
        "sample_id": s.sample_id,
        "roi_group": s.roi_group,
        "roi_id": s.roi_id,
        "patch_id": s.patch_id,
        "s1_dates": ", ".join(s.dates[("S1", t)] for t in range(4)),
        "s2_dates": ", ".join(s.dates[("S2", t)] for t in range(4)),
    })
index_df = pd.DataFrame(rows)
display(index_df.head())
display(index_df.groupby(["roi_group", "roi_id"]).size().reset_index(name="n_patches"))
index_df.to_csv(CFG.OUT_DIR / "sen12ms_asiawest_index.csv", index=False)

## 4. Reader dan Visualisasi RGB
Band Sentinel-2 umum: B2=Blue index 1, B3=Green index 2, B4=Red index 3.

In [ ]:
def read_tif(path: Path) -> np.ndarray:
    arr = tifffile.imread(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]
    if arr.ndim == 3 and arr.shape[-1] in {2, 3, 4, 13}:
        arr = np.moveaxis(arr, -1, 0)
    return arr.astype(np.float32, copy=False)


def robust_scale(arr: np.ndarray, p_low=2, p_high=98) -> np.ndarray:
    lo, hi = np.nanpercentile(arr, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)


def normalize_s2(arr: np.ndarray) -> np.ndarray:
    return np.clip(arr / 10000.0, 0, 1).astype(np.float32)


def normalize_s1(arr: np.ndarray) -> np.ndarray:
    # S1 biasanya dB sekitar -35..5. Skala konservatif ke [0,1].
    return np.clip((arr + 35.0) / 40.0, 0, 1).astype(np.float32)


def s2_rgb(s2: np.ndarray) -> np.ndarray:
    return robust_scale(np.stack([s2[3], s2[2], s2[1]], axis=-1))


def s1_vv_vh_rgb(s1: np.ndarray) -> np.ndarray:
    vv = robust_scale(s1[0])
    vh = robust_scale(s1[1])
    return np.stack([vv, vh, np.abs(vv-vh)], axis=-1)


def spectral_cloud_probability_s2(s2_norm: np.ndarray) -> np.ndarray:
    # SEN12MS band order umum: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B10,B11,B12.
    blue = s2_norm[1]
    green = s2_norm[2]
    red = s2_norm[3]
    nir = s2_norm[7] if s2_norm.shape[0] > 7 else red
    cirrus = s2_norm[10] if s2_norm.shape[0] > 10 else blue
    swir1 = s2_norm[11] if s2_norm.shape[0] > 11 else nir
    swir2 = s2_norm[12] if s2_norm.shape[0] > 12 else swir1
    rgb = np.stack([red, green, blue], axis=0)
    brightness = rgb.mean(axis=0)
    whiteness = 1.0 - rgb.std(axis=0)
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndsi = (green - swir1) / (green + swir1 + 1e-6)
    visible_cloud = brightness * np.clip(whiteness, 0, 1)
    high_band = 0.60 * cirrus + 0.25 * swir1 + 0.15 * swir2
    vegetation_penalty = np.clip((ndvi - 0.20) / 0.35, 0, 1)
    snow_penalty = np.clip((ndsi - 0.35) / 0.35, 0, 1) * np.clip((brightness - 0.20) / 0.45, 0, 1)
    prob = 0.62 * visible_cloud + 0.38 * high_band
    prob = prob * (1.0 - 0.35 * vegetation_penalty) * (1.0 - 0.25 * snow_penalty)
    return np.clip(prob, 0, 1).astype(np.float32)


def temporal_cloud_probabilities_s2(stack_norm: np.ndarray) -> np.ndarray:
    # stack_norm: T,C,H,W. Temporal contrast mengurangi false positive pada tanah/salju cerah stabil.
    spectral = np.stack([spectral_cloud_probability_s2(stack_norm[t]) for t in range(stack_norm.shape[0])], axis=0)
    blue = stack_norm[:, 1]
    green = stack_norm[:, 2]
    red = stack_norm[:, 3]
    brightness = (red + green + blue) / 3.0
    temporal_floor = np.percentile(brightness, 20, axis=0)
    temporal_median = np.median(brightness, axis=0)
    brighter_than_clear = np.clip((brightness - temporal_floor) / 0.18, 0, 1)
    brighter_than_median = np.clip((brightness - temporal_median) / 0.12, 0, 1)
    temporal = 0.65 * brighter_than_clear + 0.35 * brighter_than_median
    probs = 0.58 * spectral + 0.42 * temporal
    return np.clip(probs, 0, 1).astype(np.float32)


def refine_soft_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.clip(mask, 0, 1).astype(np.float32)
    if gaussian_filter is not None:
        mask = gaussian_filter(mask, sigma=1.0)
    return np.clip(mask, 0, 1).astype(np.float32)


def soft_cloud_mask_from_prob(prob: np.ndarray) -> np.ndarray:
    mask = np.clip((prob - CFG.MASK_LOW) / (CFG.MASK_HIGH - CFG.MASK_LOW + 1e-6), 0, 1)
    return refine_soft_mask(mask)


def cloud_probability_s2(s2: np.ndarray) -> np.ndarray:
    arr = normalize_s2(s2) if s2.max() > 1.5 else s2.astype(np.float32)
    return spectral_cloud_probability_s2(arr)


def cloud_score_s2(s2: np.ndarray) -> float:
    return float(cloud_probability_s2(s2).mean())

sample = samples[0]
s2 = read_tif(sample.s2[0])
s1 = read_tif(sample.s1[0])
s2_stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)
probs = temporal_cloud_probabilities_s2(s2_stack)
print("S2 shape/dtype/min/max:", s2.shape, s2.dtype, float(s2.min()), float(s2.max()))
print("S1 shape/dtype/min/max:", s1.shape, s1.dtype, float(s1.min()), float(s1.max()))
print("Temporal cloud prob t0 mean/max:", float(probs[0].mean()), float(probs[0].max()))
print("Soft mask t0 mean/max:", float(soft_cloud_mask_from_prob(probs[0]).mean()), float(soft_cloud_mask_from_prob(probs[0]).max()))

In [ ]:
def plot_temporal_sample(sample: SEN12Sample):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(sample.sample_id, fontsize=14)
    for t in range(4):
        s2 = read_tif(sample.s2[t])
        s1 = read_tif(sample.s1[t])
        axes[0, t].imshow(s2_rgb(s2))
        axes[0, t].set_title(f"S2 t{t} | {sample.dates[('S2', t)]}\ncloud score={cloud_score_s2(s2):.3f}")
        axes[0, t].axis("off")
        axes[1, t].imshow(s1_vv_vh_rgb(s1))
        axes[1, t].set_title(f"S1 t{t} | {sample.dates[('S1', t)]}")
        axes[1, t].axis("off")
    plt.tight_layout()
    plt.show()

for s in random.sample(samples, min(3, len(samples))):
    plot_temporal_sample(s)

## 5. Statistik Band Ringan
Sampling dibatasi agar cepat di Kaggle.

In [ ]:
def band_stats(sample_list: List[SEN12Sample], max_samples=80) -> pd.DataFrame:
    chosen = random.sample(sample_list, min(max_samples, len(sample_list)))
    rows = []
    for s in chosen:
        for sensor, paths in [("S1", s.s1), ("S2", s.s2)]:
            for t, path in paths.items():
                arr = read_tif(path)
                rows.append({
                    "sensor": sensor,
                    "time_id": t,
                    "sample_id": s.sample_id,
                    "bands": arr.shape[0],
                    "min": float(np.nanmin(arr)),
                    "p2": float(np.nanpercentile(arr, 2)),
                    "mean": float(np.nanmean(arr)),
                    "p98": float(np.nanpercentile(arr, 98)),
                    "max": float(np.nanmax(arr)),
                    "cloud_score": cloud_score_s2(arr) if sensor == "S2" else np.nan,
                })
    return pd.DataFrame(rows)

stats_df = band_stats(samples)
display(stats_df.groupby(["sensor", "time_id"])[["min", "p2", "mean", "p98", "max", "cloud_score"]].mean().round(3))
stats_df.to_csv(CFG.OUT_DIR / "eda_band_stats_sample.csv", index=False)

## 6. Baseline Cloud Removal Sederhana
Baseline awal:
- **Cloudy t0**: gambar pertama apa adanya.
- **Best-time**: pilih tanggal dengan cloud score paling rendah.
- **Pseudo ground truth / clear target**: pilih/gabung pixel dari tanggal yang paling bersih secara per-pixel.

Karena `asiaWest_n` tidak punya clear label eksplisit, kolom ground truth di notebook ini berarti **pseudo ground truth**, bukan label clear asli.

In [ ]:
def temporal_median(sample: SEN12Sample) -> np.ndarray:
    stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)
    return np.median(stack, axis=0).astype(np.float32)


def clear_pixel_composite(sample: SEN12Sample) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)  # T,C,H,W
    probs = temporal_cloud_probabilities_s2(stack)                                    # T,H,W
    best_map = np.argmin(probs, axis=0).astype(np.int64)
    weights = np.exp(-CFG.TARGET_SHARPNESS * probs).astype(np.float32) + 1e-5
    weights = weights / weights.sum(axis=0, keepdims=True)
    target_weighted = (stack * weights[:, None]).sum(axis=0).astype(np.float32)
    target_best = np.zeros_like(target_weighted)
    for t in range(stack.shape[0]):
        pick = best_map == t
        target_best[:, pick] = stack[t, :, pick].T
    # Campur best-pixel dan weighted composite: cukup tajam, tapi lebih halus dari hard best-map.
    target = (0.45 * target_best + 0.55 * target_weighted).astype(np.float32)
    cloud_t0 = soft_cloud_mask_from_prob(probs[0])
    return target, cloud_t0, probs, best_map


def clear_pixel_composite_from_stack(stack: np.ndarray, probs: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    best_map = np.argmin(probs, axis=0).astype(np.int64)
    weights = np.exp(-CFG.TARGET_SHARPNESS * probs).astype(np.float32) + 1e-5
    weights = weights / weights.sum(axis=0, keepdims=True)
    target_weighted = (stack * weights[:, None]).sum(axis=0).astype(np.float32)
    target_best = np.zeros_like(target_weighted)
    for t in range(stack.shape[0]):
        pick = best_map == t
        target_best[:, pick] = stack[t, :, pick].T
    target = (0.45 * target_best + 0.55 * target_weighted).astype(np.float32)
    cloud_t0 = soft_cloud_mask_from_prob(probs[0])
    return target, cloud_t0, best_map


def weighted_clear_composite(sample: SEN12Sample) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    target, cloud_t0, probs, _ = clear_pixel_composite(sample)
    return target, cloud_t0, probs


def best_time_s2(sample: SEN12Sample) -> Tuple[int, np.ndarray, List[float]]:
    arrs = [normalize_s2(read_tif(sample.s2[t])) for t in range(4)]
    probs = temporal_cloud_probabilities_s2(np.stack(arrs, axis=0))
    scores = [float(probs[t].mean()) for t in range(4)]
    best_t = int(np.argmin(scores))
    return best_t, arrs[best_t], scores


def np_mae(a, b, mask=None):
    err = np.abs(a - b)
    if mask is not None and mask.sum() > 0:
        err = err * mask[None]
        return float(err.sum() / (mask.sum() * a.shape[0] + 1e-8))
    return float(err.mean())


def plot_baseline(sample: SEN12Sample):
    best_t, best_arr, scores = best_time_s2(sample)
    target, cloud_t0, probs, best_map = clear_pixel_composite(sample)
    t0 = normalize_s2(read_tif(sample.s2[0]))
    median_arr = temporal_median(sample)
    fig, axes = plt.subplots(2, 6, figsize=(21, 7))
    for t in range(4):
        axes[0, t].imshow(s2_rgb(read_tif(sample.s2[t])))
        axes[0, t].set_title(f"S2 t{t} | score={scores[t]:.3f}")
        axes[0, t].axis("off")
    axes[0, 4].imshow(cloud_t0, cmap="gray", vmin=0, vmax=1)
    axes[0, 4].set_title("auto cloud mask t0")
    axes[0, 4].axis("off")
    axes[0, 5].imshow(best_map, cmap="viridis", vmin=0, vmax=3)
    axes[0, 5].set_title("best date map")
    axes[0, 5].axis("off")
    panels = [(t0, "cloudy t0"), (best_arr, f"best whole t{best_t}"), (median_arr, "median"), (target, "pseudo ground truth")]
    for ax, (arr, title) in zip(axes[1, :4], panels):
        ax.imshow(s2_rgb(arr))
        ax.set_title(title)
        ax.axis("off")
    axes[1, 4].imshow(probs[0], cmap="magma", vmin=0, vmax=1)
    axes[1, 4].set_title("cloud prob t0")
    axes[1, 4].axis("off")
    axes[1, 5].axis("off")
    plt.suptitle(sample.sample_id)
    plt.tight_layout()
    plt.show()
    print({
        "t0_mae_to_pseudo_gt": np_mae(t0, target),
        "best_mae_to_pseudo_gt": np_mae(best_arr, target),
        "median_mae_to_pseudo_gt": np_mae(median_arr, target),
        "t0_cloud_mae": np_mae(t0, target, cloud_t0),
        "best_cloud_mae": np_mae(best_arr, target, cloud_t0),
        "mask_mean": float(cloud_t0.mean()),
    })

for s in random.sample(samples, min(5, len(samples))):
    plot_baseline(s)

## 7. Dataset PyTorch Multi-Temporal
Input versi ini:
- S2: 4 time step x 13 band = 52 channel.
- S1: 4 time step x 2 band = 8 channel.
- Pseudo cloud probability: 4 channel.

Total input = 64 channel. Target = weighted clear composite S2 13 band. Soft mask t0 dipakai untuk metrik `cloud_mae`, jadi loss fokus ke pixel yang paling mungkin berawan.

In [ ]:
class SEN12TemporalDataset(Dataset):
    def __init__(self, samples: List[SEN12Sample], target_mode="weighted"):
        self.samples = samples
        self.target_mode = target_mode

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        s2_list = [normalize_s2(read_tif(s.s2[t])) for t in range(4)]
        temporal_probs = temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))
        cloud_probs = [temporal_probs[t][None] for t in range(4)]
        inputs = [*s2_list, *cloud_probs]
        if CFG.USE_S1:
            s1_list = [normalize_s1(read_tif(s.s1[t])) for t in range(4)]
            inputs.extend(s1_list)
        x = np.concatenate(inputs, axis=0).astype(np.float32)

        if self.target_mode == "weighted":
            target, cloud_mask, _ = clear_pixel_composite_from_stack(np.stack(s2_list, axis=0), temporal_probs)
        elif self.target_mode == "median":
            target = temporal_median(s)
            cloud_mask = soft_cloud_mask_from_prob(temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))[0])
        elif self.target_mode == "best_time":
            _, target, _ = best_time_s2(s)
            cloud_mask = soft_cloud_mask_from_prob(temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))[0])
        else:
            raise ValueError(self.target_mode)
        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(target.astype(np.float32)),
            "cloud_mask": torch.from_numpy(cloud_mask[None].astype(np.float32)),
            "t0": torch.from_numpy(s2_list[0].astype(np.float32)),
            "sample_id": s.sample_id,
        }

work_samples = samples
if CFG.MAX_TRAIN_SAMPLES and len(samples) > CFG.MAX_TRAIN_SAMPLES:
    work_samples = random.sample(samples, CFG.MAX_TRAIN_SAMPLES)

dataset = SEN12TemporalDataset(work_samples, target_mode="weighted")
item = dataset[0]
print(item["x"].shape, item["y"].shape, item["cloud_mask"].shape, item["sample_id"])
INPUT_CHANNELS = item["x"].shape[0]
print("INPUT_CHANNELS:", INPUT_CHANNELS)

In [ ]:
n = len(dataset)
n_train = int(n * CFG.TRAIN_RATIO)
n_val = int(n * CFG.VAL_RATIO)
n_test = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(SEED))


def subset_base_indices(subset):
    return list(subset.indices) if hasattr(subset, "indices") else list(range(len(subset)))


def compute_subset_cloud_means(subset, name="train"):
    """Hitung rata-rata auto mask per sample untuk oversampling dan bucket analysis."""
    base_indices = subset_base_indices(subset)
    base_dataset = subset.dataset if hasattr(subset, "dataset") else subset
    rows = []
    for local_i, base_i in enumerate(tqdm(base_indices, desc=f"cloud mean scan {name}", leave=False)):
        item = base_dataset[int(base_i)]
        rows.append({
            "local_index": local_i,
            "base_index": int(base_i),
            "sample_id": item["sample_id"],
            "cloud_mean": float(item["cloud_mask"].mean()),
        })
    return pd.DataFrame(rows)

train_cloud_df = compute_subset_cloud_means(train_ds, "train")
val_cloud_df = compute_subset_cloud_means(val_ds, "val")
train_cloud_df.to_csv(CFG.OUT_DIR / "train_cloud_sampling_weights.csv", index=False)
val_cloud_df.to_csv(CFG.OUT_DIR / "val_cloud_distribution.csv", index=False)

def make_train_sampler(cloud_df):
    cloud = cloud_df["cloud_mean"].to_numpy(dtype=np.float32)
    # Sample berawan sedang/berat muncul lebih sering agar model tidak hanya bagus di awan ringan.
    weights = 1.0 + 2.0 * cloud
    weights += 3.0 * (cloud >= CFG.HEAVY_CLOUD_THRESHOLD).astype(np.float32)
    weights += 2.0 * (cloud >= 0.45).astype(np.float32)
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
        generator=torch.Generator().manual_seed(SEED),
    ), weights

loader_kwargs = dict(num_workers=CFG.NUM_WORKERS, pin_memory=torch.cuda.is_available())
if CFG.NUM_WORKERS > 0:
    loader_kwargs.update(prefetch_factor=2, persistent_workers=True)

if CFG.HEAVY_CLOUD_OVERSAMPLE:
    train_sampler, train_sampling_weights = make_train_sampler(train_cloud_df)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=train_sampler, **loader_kwargs)
else:
    train_sampling_weights = np.ones(len(train_ds), dtype=np.float32)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, **loader_kwargs)

val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, **loader_kwargs)

print("split:", len(train_ds), len(val_ds), len(test_ds))
print("train cloud mean summary:")
display(train_cloud_df["cloud_mean"].describe().to_frame().T)
print("heavy train samples:", int((train_cloud_df["cloud_mean"] >= CFG.HEAVY_CLOUD_THRESHOLD).sum()))
print("oversampling:", CFG.HEAVY_CLOUD_OVERSAMPLE, "weight min/max:", float(np.min(train_sampling_weights)), float(np.max(train_sampling_weights)))
batch = next(iter(train_loader))
print("batch x/y:", batch["x"].shape, batch["y"].shape)

## 8. Compact Multi-Temporal U-Net / ResUNet
Model tetap U-Net + residual block. Perubahan kecil: upsampling bilinear dipakai untuk mengurangi artefak checkerboard dibanding transpose convolution.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), ConvBlock(in_ch, out_ch))
    def forward(self, x):
        return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class MultiTemporalResUNet(nn.Module):
    def __init__(self, in_channels=64, out_channels=13, base=32):
        super().__init__()
        self.inc = ConvBlock(in_channels, base)
        self.down1 = Down(base, base*2)
        self.down2 = Down(base*2, base*4)
        self.down3 = Down(base*4, base*8)
        self.up1 = Up(base*8, base*4, base*4)
        self.up2 = Up(base*4, base*2, base*2)
        self.up3 = Up(base*2, base, base)
        self.outc = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)
        return torch.sigmoid(self.outc(x))

model = MultiTemporalResUNet(in_channels=INPUT_CHANNELS, base=CFG.BASE_CHANNELS).to(DEVICE)
with torch.no_grad():
    yhat = model(batch["x"].to(DEVICE))
print("output:", yhat.shape)
print("params:", sum(p.numel() for p in model.parameters()))

# ==================== GAN & DISCRIMINATOR SETUP ====================
import torch.nn.utils.spectral_norm as spectral_norm

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            spectral_norm(nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential(
            spectral_norm(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))

class ResNetDiscriminator(nn.Module):
    def __init__(self, in_channels=13, base=64):
        super().__init__()
        self.init_conv = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels, base, 7, stride=2, padding=3, bias=False)),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = ResidualBlock(base, base)
        self.layer2 = ResidualBlock(base, base*2, stride=2)
        self.layer3 = ResidualBlock(base*2, base*4, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(base*4, 1)

    def forward(self, x):
        x = self.init_conv(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# GAN Initialization
netD = ResNetDiscriminator(in_channels=13, base=CFG.BASE_CHANNELS).to(DEVICE)
print('Discriminator model initialized successfully.')


## 9. Training Monitor, Heavy-Cloud Loss, dan Bucket Metrics
Versi ini tetap memakai ResUNet/U-Net, tetapi training strategy dibuat lebih fokus ke awan berat:

- Area luar mask tetap copy dari cloudy t0.
- Sample dengan `cloud_mean` tinggi di-oversample saat training.
- Loss area awan diberi bobot lebih besar pada pixel mask tinggi.
- Evaluasi ditambah bucket `light / medium / heavy cloud MAE`.
- Checkpoint terbaik tetap dipilih dari `model_cloud_mae`, dengan `model_heavy_cloud_mae` sebagai indikator tambahan untuk awan tebal.

In [ ]:
def mae(pred, target):
    return F.l1_loss(pred, target)

def masked_mae(pred, target, mask):
    diff = torch.abs(pred - target) * mask
    return diff.sum() / (mask.sum() * pred.shape[1] + 1e-8)

def weighted_masked_mae(pred, target, mask, weight):
    diff = torch.abs(pred - target) * mask * weight
    return diff.sum() / ((mask * weight).sum() * pred.shape[1] + 1e-8)

def bucket_mae(pred, target, bucket_mask):
    diff = torch.abs(pred - target) * bucket_mask
    val = diff.sum() / (bucket_mask.sum() * pred.shape[1] + 1e-8)
    return torch.where(bucket_mask.sum() > 0, val, torch.tensor(float('nan'), device=pred.device))

def gradient_mae(pred, target, mask):
    dy_pred = torch.abs(pred[:, :, 1:, :] - pred[:, :, :-1, :])
    dy_target = torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :])
    dx_pred = torch.abs(pred[:, :, :, 1:] - pred[:, :, :, :-1])
    dx_target = torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1])
    
    mask_y = mask[:, :, 1:, :]
    mask_x = mask[:, :, :, 1:]
    
    loss_y = torch.abs(dy_pred - dy_target) * mask_y
    loss_x = torch.abs(dx_pred - dx_target) * mask_x
    
    total_loss = loss_y.sum() / (mask_y.sum() * pred.shape[1] + 1e-8) + loss_x.sum() / (mask_x.sum() * pred.shape[1] + 1e-8)
    return total_loss

def rmse(pred, target):
    return torch.sqrt(F.mse_loss(pred, target))

def psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target)
    return 20 * torch.log10(torch.tensor(max_val, device=pred.device)) - 10 * torch.log10(mse + 1e-8)

def apply_copy_outside_mask(raw_pred, t0, mask):
    return raw_pred * mask + t0 * (1.0 - mask)

def heavy_cloud_weight(mask):
    return 1.0 + CFG.HEAVY_LOSS_BOOST * mask + CFG.HEAVY_BINARY_BOOST * (mask >= CFG.HEAVY_PIXEL_THRESHOLD).float()

def cloud_bucket_masks(mask):
    light = (mask > 0.0) & (mask < 0.33)
    medium = (mask >= 0.33) & (mask < 0.66)
    heavy = (mask >= 0.66)
    return light.float(), medium.float(), heavy.float()

def scalar_or_none(x):
    return float(x.detach().cpu()) if not torch.isnan(x) else None

def evaluate(loader, label="val"):
    model.eval()
    netD.eval()
    metrics = {
        "model_mae": [], "model_cloud_mae": [], "model_light_cloud_mae": [],
        "model_medium_cloud_mae": [], "model_heavy_cloud_mae": [],
        "raw_cloud_mae": [], "t0_mae": [], "t0_cloud_mae": [],
        "t0_light_cloud_mae": [], "t0_medium_cloud_mae": [], "t0_heavy_cloud_mae": [],
        "rmse": [], "psnr": [], "cloud_fraction": [], "heavy_fraction": []
    }
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            mask = batch["cloud_mask"].to(DEVICE)
            t0 = batch["t0"].to(DEVICE)
            
            raw = model(x)
            pred = apply_copy_outside_mask(raw, t0, mask)
            
            light_m, medium_m, heavy_m = cloud_bucket_masks(mask)
            
            metrics["model_mae"].append(float(F.l1_loss(pred, y).cpu()))
            metrics["model_cloud_mae"].append(float(masked_mae(pred, y, mask).cpu()))
            metrics["model_light_cloud_mae"].append(scalar_or_none(bucket_mae(pred, y, light_m)))
            metrics["model_medium_cloud_mae"].append(scalar_or_none(bucket_mae(pred, y, medium_m)))
            metrics["model_heavy_cloud_mae"].append(scalar_or_none(bucket_mae(pred, y, heavy_m)))
            metrics["raw_cloud_mae"].append(float(masked_mae(raw, y, mask).cpu()))
            
            metrics["t0_mae"].append(float(F.l1_loss(t0, y).cpu()))
            metrics["t0_cloud_mae"].append(float(masked_mae(t0, y, mask).cpu()))
            metrics["t0_light_cloud_mae"].append(scalar_or_none(bucket_mae(t0, y, light_m)))
            metrics["t0_medium_cloud_mae"].append(scalar_or_none(bucket_mae(t0, y, medium_m)))
            metrics["t0_heavy_cloud_mae"].append(scalar_or_none(bucket_mae(t0, y, heavy_m)))
            
            metrics["rmse"].append(float(rmse(pred, y).cpu()))
            metrics["psnr"].append(float(psnr(pred, y).cpu()))
            metrics["cloud_fraction"].append(float(mask.mean().cpu()))
            metrics["heavy_fraction"].append(float(heavy_m.mean().cpu()))
            
    res = {}
    for k, vals in metrics.items():
        clean = [v for v in vals if v is not None]
        res[k] = float(np.mean(clean)) if clean else 0.0
    return res

def train_one_epoch(loader, epoch, total_epochs):
    model.train()
    netD.train()
    losses = []
    d_losses = []
    heavy_losses = []
    iterator = tqdm(loader, desc=f"epoch {epoch:03d}/{total_epochs:03d}", leave=True)
    for batch in iterator:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mask = batch["cloud_mask"].to(DEVICE)
        t0 = batch["t0"].to(DEVICE)
        
        # 1. Update Discriminator
        optimizerD.zero_grad()
        # Real
        out_real = netD(y)
        loss_D_real = criterion_GAN(out_real, torch.ones_like(out_real))
        # Fake
        raw = model(x)
        pred = apply_copy_outside_mask(raw, t0, mask)
        out_fake = netD(pred.detach())
        loss_D_fake = criterion_GAN(out_fake, torch.zeros_like(out_fake))
        
        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizerD.step()
        d_losses.append(float(loss_D.detach().cpu()))
        
        # 2. Update Generator
        optimizer.zero_grad(set_to_none=True)
        out_fake_G = netD(pred)
        loss_G_adv = criterion_GAN(out_fake_G, torch.ones_like(out_fake_G))
        
        weight = heavy_cloud_weight(mask)
        whole = F.l1_loss(pred, y)
        cloud_plain = masked_mae(pred, y, mask)
        cloud_weighted = weighted_masked_mae(pred, y, mask, weight)
        raw_cloud_weighted = weighted_masked_mae(raw, y, mask, weight)
        copy_penalty = masked_mae(pred, t0, 1.0 - mask)
        grad = gradient_mae(pred, y, mask)
        _, _, heavy_m = cloud_bucket_masks(mask)
        heavy_only = bucket_mae(pred, y, heavy_m)
        heavy_term = torch.where(torch.isnan(heavy_only), cloud_plain, heavy_only)
        
        loss = (
            0.15 * whole
            + 1.15 * cloud_weighted
            + 0.20 * raw_cloud_weighted
            + 0.25 * copy_penalty
            + 0.10 * grad
            + 0.15 * heavy_term
            + 0.05 * loss_G_adv
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
        heavy_losses.append(0.0 if bool(torch.isnan(heavy_only).detach().cpu()) else float(heavy_only.detach().cpu()))
        
        iterator.set_postfix(loss_G=np.mean(losses), loss_D=np.mean(d_losses))
        
    return float(np.mean(losses)), float(np.mean(d_losses)), float(np.mean(heavy_losses))

def save_training_plot(history_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train G loss")
    axes[0].plot(history_df["epoch"], history_df["train_cloud_loss"], marker="o", label="train D loss")
    axes[0].plot(history_df["epoch"], history_df["val_model_mae"], marker="x", label="val model MAE")
    axes[0].set_title("Training Loss & Val MAE")
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history_df["epoch"], history_df["val_model_cloud_mae"], marker="o", label="val cloud MAE")
    axes[1].plot(history_df["epoch"], history_df["val_model_heavy_cloud_mae"], marker="x", label="val heavy cloud MAE")
    axes[1].set_title("Validation Cloud MAE")
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(CFG.OUT_DIR / "training_curves.png", dpi=150)
    plt.close()

# Optimizer, Schedulers and GAN Loss Definitions
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=CFG.LR * 0.1)

optimizerD = torch.optim.AdamW(netD.parameters(), lr=CFG.LR, weight_decay=1e-4)
schedulerD = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerD, T_max=CFG.EPOCHS, eta_min=CFG.LR * 0.1)
criterion_GAN = nn.BCEWithLogitsLoss()

print("strategy: hard/soft copy outside mask; spectral-normalized ResNet-18 GAN; heavy-cloud oversampling")

if CFG.RUN_TRAINING:
    history = []
    best_cloud_mae = float("inf")
    
    for epoch in range(1, CFG.EPOCHS + 1):
        train_loss, train_d_loss, train_heavy_loss = train_one_epoch(train_loader, epoch, CFG.EPOCHS)
        val_metrics = evaluate(val_loader, label="val")
        
        log_row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_cloud_loss": train_d_loss,
            "train_heavy_loss": train_heavy_loss,
            "lr": optimizer.param_groups[0]["lr"],
        }
        log_row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(log_row)
        
        # Print metrics
        print(f"Epoch {epoch:02d} | Train G: {train_loss:.5f} | Train D: {train_d_loss:.5f} | Val Cloud MAE: {val_metrics['model_cloud_mae']:.5f} | PSNR: {val_metrics['psnr']:.2f} dB")
        
        # Checkpoint
        if val_metrics["model_cloud_mae"] < best_cloud_mae:
            best_cloud_mae = val_metrics["model_cloud_mae"]
            torch.save(model.state_dict(), CFG.OUT_DIR / "best_multitemporal_resunet_hardmask.pth")
            torch.save(netD.state_dict(), CFG.OUT_DIR / "best_resnet_discriminator.pth")
            print(f"  ==> Saved best model weights with cloud MAE {best_cloud_mae:.6f}")
            
        torch.save(model.state_dict(), CFG.OUT_DIR / "last_multitemporal_resunet_hardmask.pth")
        
        # Step schedulers
        scheduler.step()
        schedulerD.step()
        
        # Save history periodically
        history_df = pd.DataFrame(history)
        history_df.to_csv(CFG.OUT_DIR / "training_history.csv", index=False)
        save_training_plot(history_df)


## 10. Final Output dan Perbandingan RGB
Bagian ini adalah visual utama untuk laporan/evaluasi akhir.

Kolom yang ditampilkan:
- **Input Cloudy**: citra Sentinel-2 waktu t0 yang masih berawan.
- **Pseudo Ground Truth**: target clear hasil komposit multi-temporal, bukan label manual asli.
- **Generated Best**: output dari checkpoint terbaik berdasarkan `model_cloud_mae`.
- **RGB Difference**: selisih absolut RGB antara generated dan pseudo ground truth; makin gelap makin mirip.
- **Auto Mask**: area yang dianggap berawan dan menjadi fokus perbaikan model.

Catatan interpretasi: generated dianggap mendekati pseudo ground truth jika `RGB_MAE` kecil, `Masked_RGB_MAE` kecil, dan `RGB_PSNR` tinggi. Hasil yang baik tetap bisa terlihat sedikit smooth/blur karena model belajar rekonstruksi dari pseudo target, bukan ground truth clear manual.

In [ ]:
@torch.no_grad()
def rgb_tensor_np(s2_batch: np.ndarray) -> np.ndarray:
    """Return normalized RGB channels as N,H,W,3 from S2 N,C,H,W in [0,1]."""
    return np.clip(np.stack([s2_batch[:, 3], s2_batch[:, 2], s2_batch[:, 1]], axis=-1), 0, 1).astype(np.float32)


def rgb_metrics_np(pred_s2: np.ndarray, target_s2: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    pred_rgb = rgb_tensor_np(pred_s2)
    target_rgb = rgb_tensor_np(target_s2)
    diff = np.abs(pred_rgb - target_rgb)
    mask_hw = np.clip(mask[:, 0], 0, 1).astype(np.float32)
    mse = float(np.mean((pred_rgb - target_rgb) ** 2))
    masked_denom = float(mask_hw.sum() * 3.0 + 1e-8)
    masked_rgb_mae = float((diff * mask_hw[..., None]).sum() / masked_denom)
    return {
        "rgb_mae": float(diff.mean()),
        "masked_rgb_mae": masked_rgb_mae,
        "rgb_mse": mse,
        "rgb_psnr": float(20 * np.log10(1.0) - 10 * np.log10(mse + 1e-8)),
        "mask_mean": float(mask_hw.mean()),
    }


def choose_visual_batch(loader, max_items=4, search_batches=20):
    """Pilih contoh informatif: ada awan cukup banyak, tapi tidak semuanya full mask."""
    candidates = []
    for bi, batch in enumerate(loader):
        masks = batch["cloud_mask"].numpy()
        for i in range(masks.shape[0]):
            frac = float(masks[i, 0].mean())
            # Prioritas: mask sedang-tinggi agar perbaikan terlihat, tapi hindari semua putih jika ada pilihan.
            score = -abs(frac - 0.45)
            if frac > 0.08:
                candidates.append((score, frac, bi, i, batch))
        if bi + 1 >= search_batches:
            break
    if not candidates:
        return next(iter(loader))
    candidates.sort(reverse=True, key=lambda x: x[0])
    picked = candidates[:max_items]
    # Agar sederhana dan cepat, pakai item terpilih dari kandidat individual, lalu gabungkan manual.
    rows = []
    for _, _, _, i, batch in picked:
        rows.append({k: (v[i:i+1] if torch.is_tensor(v) else [v[i]]) for k, v in batch.items()})
    merged = {}
    for k in rows[0].keys():
        if torch.is_tensor(rows[0][k]):
            merged[k] = torch.cat([r[k] for r in rows], dim=0)
        else:
            merged[k] = sum((r[k] for r in rows), [])
    return merged


@torch.no_grad()
def final_rgb_comparison(loader, max_items=4, search_batches=20):
    model.eval()
    checkpoint_info = {}
    if BEST_PATH.exists():
        checkpoint = torch.load(BEST_PATH, map_location=DEVICE)
        state = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
        model.load_state_dict(state)
        checkpoint_info = checkpoint if isinstance(checkpoint, dict) else {}
        print("Loaded best checkpoint:", BEST_PATH)
        print("Best epoch:", checkpoint_info.get("epoch"))
    else:
        print("WARNING: BEST_PATH belum ada, memakai bobot model saat ini.")

    batch = choose_visual_batch(loader, max_items=max_items, search_batches=search_batches)
    x = batch["x"].to(DEVICE)
    y = batch["y"].cpu().numpy()
    t0_t = batch["t0"].to(DEVICE)
    t0 = batch["t0"].cpu().numpy()
    mask_t = batch["cloud_mask"].to(DEVICE)
    mask = batch["cloud_mask"].cpu().numpy()
    sample_ids = batch.get("sample_id", [f"sample_{i}" for i in range(x.shape[0])])

    raw = model(x)
    generated = apply_copy_outside_mask(raw, t0_t, mask_t).cpu().numpy()

    input_rgb = rgb_tensor_np(t0)
    target_rgb = rgb_tensor_np(y)
    generated_rgb = rgb_tensor_np(generated)
    rgb_diff = np.abs(generated_rgb - target_rgb)

    rows = []
    for i in range(generated.shape[0]):
        m = rgb_metrics_np(generated[i:i+1], y[i:i+1], mask[i:i+1])
        m.update({"sample_id": sample_ids[i]})
        rows.append(m)
    metrics_df = pd.DataFrame(rows)
    display(metrics_df[["sample_id", "rgb_mae", "masked_rgb_mae", "rgb_psnr", "mask_mean"]])

    n = generated.shape[0]
    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)
    for i in range(n):
        axes[i, 0].imshow(input_rgb[i])
        axes[i, 0].set_title(f"Input Cloudy\n{sample_ids[i]}")
        axes[i, 1].imshow(target_rgb[i])
        axes[i, 1].set_title("Pseudo Ground Truth")
        axes[i, 2].imshow(generated_rgb[i])
        axes[i, 2].set_title("Generated Best")
        axes[i, 3].imshow(np.clip(rgb_diff[i] * 5.0, 0, 1))
        axes[i, 3].set_title(f"RGB Difference x5\nMAE={rows[i]['rgb_mae']:.4f}")
        axes[i, 4].imshow(mask[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[i, 4].set_title(f"Auto Mask\nmean={rows[i]['mask_mean']:.3f}")
        for ax in axes[i]:
            ax.axis("off")
    plt.tight_layout()

    out_img = CFG.OUT_DIR / "final_rgb_input_gt_generated_comparison.png"
    out_csv = CFG.OUT_DIR / "final_rgb_metrics.csv"
    out_json = CFG.OUT_DIR / "final_rgb_metrics.json"
    fig.savefig(out_img, dpi=180, bbox_inches="tight")
    metrics_df.to_csv(out_csv, index=False)
    out_json.write_text(json.dumps({
        "checkpoint": str(BEST_PATH),
        "best_epoch": checkpoint_info.get("epoch"),
        "metrics": rows,
        "interpretation": "Generated mendekati pseudo ground truth jika RGB_MAE dan Masked_RGB_MAE kecil, RGB_PSNR tinggi. Sisa blur/detail halus masih wajar karena target adalah pseudo ground truth multi-temporal."
    }, indent=2), encoding="utf-8")

    print("Saved final RGB comparison:", out_img)
    print("Saved RGB metrics:", out_csv)
    print("Saved RGB metrics JSON:", out_json)
    print("Kesimpulan singkat: generated sudah dibandingkan langsung dengan pseudo ground truth pada kanal RGB. Lihat RGB Difference: area makin gelap berarti makin mirip.")
    plt.show()
    return metrics_df

final_rgb_metrics_df = final_rgb_comparison(val_loader, max_items=4, search_batches=20)

## 11. Mode Uji Gambar Baru
Cell ini untuk skenario ujian: input tidak punya mask. Notebook membuat mask otomatis dari citra. Untuk hasil terbaik tetap gunakan 4 waktu S2 + S1; single RGB hanya mode demo terbatas.

In [ ]:
@torch.no_grad()
def make_auto_mask_for_s2_times(s2_times):
    """s2_times: list/array 4 item, masing-masing shape (13,H,W) normalized [0,1]."""
    stack = np.stack(s2_times, axis=0).astype(np.float32)
    probs = temporal_cloud_probabilities_s2(stack)
    return soft_cloud_mask_from_prob(probs[0]), probs


def explain_single_image_limit():
    print("Mode utama model: 4 waktu Sentinel-2 + 4 waktu Sentinel-1.")
    print("Jika hanya 1 RGB/foto satelit biasa: mask bisa dibuat otomatis, tetapi hasil cloud removal tidak sekuat multi-temporal.")
    print("Untuk ujian, sebut ini sebagai mode demo/inference terbatas, bukan evaluasi utama.")

explain_single_image_limit()

## Kesimpulan Versi Ini
- Data `asiaWest_n` valid: 1920 sample, S2 13 band, S1 2 band, 4 waktu.
- Mask hybrid: spectral + temporal, threshold lebih longgar agar awan tipis tetap ikut dievaluasi.
- Pseudo ground truth hybrid: best-pixel + weighted composite, jadi lebih tajam dari median tapi tidak terlalu noisy.
- Strategi final: ResUNet hanya mengubah area mask; luar mask copy dari cloudy t0.
- Versi ini menambah **heavy-cloud weighted loss**, **oversampling sample berawan berat**, dan metrik bucket `light / medium / heavy cloud MAE`.
- Checkpoint yang dipakai untuk visual akhir adalah **best checkpoint**, bukan epoch terakhir.
- Hasil akhir dinilai dari `cloud_mae`, `heavy_cloud_mae`, `RGB_MAE`, `Masked_RGB_MAE`, `RGB_PSNR`, dan visual `Input Cloudy / Pseudo Ground Truth / Generated / RGB Difference / Auto Mask`.
- Untuk foto satelit random tanpa mask: notebook bisa membuat mask otomatis. Namun model utama tetap ideal untuk multi-temporal S2+S1, bukan single RGB biasa.